In [2]:
from pathlib import Path
import json
import pandas as pd
from typing import Dict

# --- Set these two parameters ---
RESULTS_FOLDER = r"M:\\Projects\\Cost Model\\HiperSim\\valuewind\\results\\HKN_TI_boost_test"
TABLE_NAME = "valuation_metrics"  # e.g., 'valuation_metrics' or 'valuation_cashflows'

RESULTS_FOLDER = Path(RESULTS_FOLDER)
assert RESULTS_FOLDER.exists(), f"Results folder not found: {RESULTS_FOLDER}"
SCENARIOS_PATH = RESULTS_FOLDER / "scenarios.json"
assert SCENARIOS_PATH.exists(), f"scenarios.json not found at: {SCENARIOS_PATH}"

# Read scenarios.json
with open(SCENARIOS_PATH, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

if isinstance(scenarios, dict) and "scenarios" in scenarios:
    scenarios = scenarios["scenarios"]

assert isinstance(scenarios, list), "Expected a list of scenarios in scenarios.json"
print(f"Loaded {len(scenarios)} scenarios from {SCENARIOS_PATH}")
scenarios[:2]  # preview first two


# Load the parquet files into memory
RESULTS: Dict[str, pd.DataFrame] = {}
missing = []

for sc in scenarios:
    sid = str(sc.get("scenario_id", ""))
    if not sid:
        continue
    parquet_path = RESULTS_FOLDER / f"{TABLE_NAME}_df_{sid}.parquet"
    if parquet_path.exists():
        try:
            RESULTS[sid] = pd.read_parquet(parquet_path)
        except Exception as e:
            missing.append((sid, str(parquet_path), f"Read error: {e}"))
    else:
        missing.append((sid, str(parquet_path), "Missing file"))

print(f"Loaded {len(RESULTS)} DataFrame(s).")
if missing:
    print("\nFiles not loaded:")
    for m in missing:
        print(" - scenario_id=", m[0], " path=", m[1], " reason=", m[2])

        # Quick glance at the loaded dataframes (keys and row counts)
summary = {sid: (df.shape[0], df.shape[1]) for sid, df in RESULTS.items()}
summary

Loaded 21 scenarios from M:\Projects\Cost Model\HiperSim\valuewind\results\HKN_TI_boost_test\scenarios.json
Loaded 21 DataFrame(s).


{'cb0949be84': (1, 3),
 'b80b3b8ac8': (1, 3),
 'ef193de0be': (1, 3),
 '8cdd3eb18b': (1, 3),
 'a17520c34f': (1, 3),
 '71df0a3df0': (1, 3),
 '637aa159a5': (1, 3),
 'cd4757c8d5': (1, 3),
 '2f4763f2b4': (1, 3),
 'a9a4b0b1c9': (1, 3),
 '606eef0556': (1, 3),
 '62aeca2452': (1, 3),
 '3d256f7b12': (1, 3),
 '4773c6264f': (1, 3),
 '184adba8d0': (1, 3),
 '40666b4da5': (1, 3),
 'b7df94a656': (1, 3),
 '599ba71d00': (1, 3),
 '0858296b17': (1, 3),
 'bef9489a49': (1, 3),
 '07e1200cec': (1, 3)}

In [14]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Load scenarios.json if not already present ---
if "scenarios" not in globals():
    SCENARIOS_PATH = Path(RESULTS_FOLDER) / "scenarios.json"
    with open(SCENARIOS_PATH, "r", encoding="utf-8") as f:
        scenarios = json.load(f)
    if isinstance(scenarios, dict) and "scenarios" in scenarios:
        scenarios = scenarios["scenarios"]

# --- Build helpers to find scenario metadata and metrics from each DF ---
def _col_ci(df, name_options=("NPV",)):
    """Return the first column name matching any option (case-insensitive)."""
    lower_map = {c.lower(): c for c in df.columns}
    for opt in name_options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    # fallback: look for substring match
    for c in df.columns:
        lc = c.lower()
        if any(opt.lower() in lc for opt in name_options):
            return c
    return None

def _extract_metric(df, options):
    """Pick a numeric scalar from the first matching column."""
    if df is None or df.empty:
        return None
    col = _col_ci(df, options)
    if col is None:
        return None
    s = df[col]
    # Try last non-null numeric value
    try:
        s_numeric = s[pd.to_numeric(s, errors="coerce").notna()]
        if not s_numeric.empty:
            return float(s_numeric.iloc[-1])
    except Exception:
        pass
    # Fallback: if already numeric-like
    try:
        return float(s.iloc[-1])
    except Exception:
        return None

# --- Which schemes to consider ---
SCHEMES = ["FiT", "CfD", "Market"]

# If you don't already have ext_map from above, (re)build it:
ext_map = {}
for sc in scenarios:
    sid = str(sc.get("scenario_id", ""))
    ov = sc.get("overrides", {})
    ext = ov.get("WindFarm_overrides.external_response_path", None)
    scheme = ov.get("Revenue_overrides.scheme_type", None)
    if not sid or not ext or scheme not in SCHEMES:
        continue
    d = ext_map.setdefault(ext, {})
    d[scheme] = sid

def _get_metrics_for_sid(sid):
    df = RESULTS.get(sid)
    return {
        "NPV":  _extract_metric(df, ("NPV", "net_present_value", "npv")),
        "LCOE": _extract_metric(df, ("LCOE", "lcoe")),
    }

# ------------- Figure 1: Absolute NPV per scheme -------------
fig_npv = go.Figure()

for ext_path, schemes in ext_map.items():
    xs, ys = [], []
    for scheme in SCHEMES:
        sid = schemes.get(scheme)
        if not sid:
            continue
        m = _get_metrics_for_sid(sid)
        if m["NPV"] is None:
            continue
        xs.append(scheme)
        ys.append(m["NPV"])

    if xs:
        label = Path(ext_path).name or str(ext_path)
        fig_npv.add_trace(go.Scatter(
            x=xs,
            y=ys,
            mode="markers",
            marker=dict(size=10, symbol="circle"),
            name=label,
            hovertemplate="Scheme: %{x}<br>NPV: %{y}<extra>" + label + "</extra>",
            showlegend=True,
        ))

fig_npv.update_layout(
    title="Absolute NPV by Scheme",
    xaxis_title="Scheme",
    yaxis_title="NPV",
    template="plotly_white",
    width=900,
    height=600,
    legend_title_text="External response",
    xaxis=dict(categoryorder="array", categoryarray=SCHEMES),
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
)
fig_npv.show()

# ------------- Figure 2: Absolute LCOE per scheme -------------
fig_lcoe = go.Figure()

for ext_path, schemes in ext_map.items():
    xs, ys = [], []
    for scheme in SCHEMES:
        sid = schemes.get(scheme)
        if not sid:
            continue
        m = _get_metrics_for_sid(sid)
        if m["LCOE"] is None:
            continue
        xs.append(scheme)
        ys.append(m["LCOE"])

    if xs:
        label = Path(ext_path).name or str(ext_path)
        fig_lcoe.add_trace(go.Scatter(
            x=xs,
            y=ys,
            mode="markers",
            marker=dict(size=10, symbol="diamond"),
            name=label,
            hovertemplate="Scheme: %{x}<br>LCOE: %{y}<extra>" + label + "</extra>",
            showlegend=True,
        ))

fig_lcoe.update_layout(
    title="Absolute LCOE by Scheme",
    xaxis_title="Scheme",
    yaxis_title="LCOE",
    template="plotly_white",
    width=900,
    height=600,
    legend_title_text="External response",
    xaxis=dict(categoryorder="array", categoryarray=SCHEMES),
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
)
fig_lcoe.show()


In [12]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Load scenarios.json if not already present ---
if "scenarios" not in globals():
    SCENARIOS_PATH = Path(RESULTS_FOLDER) / "scenarios.json"
    with open(SCENARIOS_PATH, "r", encoding="utf-8") as f:
        scenarios = json.load(f)
    if isinstance(scenarios, dict) and "scenarios" in scenarios:
        scenarios = scenarios["scenarios"]

# --- Build helpers to find scenario metadata and metrics from each DF ---
def _col_ci(df, name_options=("NPV",)):
    """Return the first column name matching any option (case-insensitive)."""
    lower_map = {c.lower(): c for c in df.columns}
    for opt in name_options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    # fallback: look for substring match
    for c in df.columns:
        lc = c.lower()
        if any(opt.lower() in lc for opt in name_options):
            return c
    return None

def _extract_metric(df, options):
    """Pick a numeric scalar from the first matching column."""
    if df is None or df.empty:
        return None
    col = _col_ci(df, options)
    if col is None:
        return None
    s = df[col]
    # Try last non-null numeric value
    try:
        s_numeric = s[pd.to_numeric(s, errors="coerce").notna()]
        if not s_numeric.empty:
            return float(s_numeric.iloc[-1])
    except Exception:
        pass
    # Fallback: if already numeric-like
    try:
        return float(s.iloc[-1])
    except Exception:
        return None

# --- Which schemes to consider ---
SCHEMES = ["FiT", "CfD", "Market"]
CATS = SCHEMES + ["lcoe"]  # add single LCOE column

def _get_metrics_for_sid(sid):
    df = RESULTS.get(sid)
    return {
        "NPV":  _extract_metric(df, ("NPV", "net_present_value", "npv")),
        "LCOE": _extract_metric(df, ("LCOE", "lcoe")),
    }

# ------------- Single figure: NPV (y) + LCOE from FiT (y2) -------------
fig = go.Figure()

for ext_path, schemes in ext_map.items():
    label = Path(ext_path).name or str(ext_path)

    # NPV per scheme (primary y-axis)
    xs_npv, ys_npv = [], []
    for scheme in SCHEMES:
        sid = schemes.get(scheme)
        if not sid:
            continue
        m = _get_metrics_for_sid(sid)
        if m["NPV"] is None:
            continue
        xs_npv.append(scheme)
        ys_npv.append(m["NPV"])

    if xs_npv:
        fig.add_trace(go.Scatter(
            x=xs_npv,
            y=ys_npv,
            mode="markers",
            marker=dict(size=10, symbol="circle"),
            name=f"{label} — NPV",
            hovertemplate="Scheme: %{x}<br>NPV: %{y}<extra>" + label + " — NPV</extra>",
            showlegend=True,
        ))

    # LCOE point from FiT only (secondary y-axis)
    sid_fit = schemes.get("FiT")
    if sid_fit:
        m_fit = _get_metrics_for_sid(sid_fit)
        lcoe_val = m_fit["LCOE"]
        if lcoe_val is not None:
            fig.add_trace(go.Scatter(
                x=["lcoe"],
                y=[lcoe_val],
                mode="markers",
                marker=dict(size=10, symbol="diamond"),
                name=f"{label} — LCOE",
                hovertemplate="Metric: LCOE<br>LCOE: %{y}<extra>" + label + " — LCOE</extra>",
                yaxis="y2",
                showlegend=True,
            ))

fig.update_layout(
    title="NPV by Scheme (left) + LCOE from FiT (right)",
    xaxis_title="Scheme / Metric",
    yaxis_title="NPV",
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis2=dict(title="LCOE", overlaying="y", side="right", showgrid=False),
    template="plotly_white",
    width=900,
    height=600,
    legend_title_text="External response",
    xaxis=dict(categoryorder="array", categoryarray=CATS),
)

fig.show()


In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Load scenarios.json if not already present ---
if "scenarios" not in globals():
    SCENARIOS_PATH = Path(RESULTS_FOLDER) / "scenarios.json"
    with open(SCENARIOS_PATH, "r", encoding="utf-8") as f:
        scenarios = json.load(f)
    if isinstance(scenarios, dict) and "scenarios" in scenarios:
        scenarios = scenarios["scenarios"]

# --- Build helpers to find scenario metadata and metrics from each DF ---
def _col_ci(df, name_options=("NPV",)):
    """Return the first column name matching any option (case-insensitive)."""
    lower_map = {c.lower(): c for c in df.columns}
    for opt in name_options:
        if opt.lower() in lower_map:
            return lower_map[opt.lower()]
    # fallback: look for substring match
    for c in df.columns:
        lc = c.lower()
        if any(opt.lower() in lc for opt in name_options):
            return c
    return None

def _extract_metric(df, options):
    """Pick a numeric scalar from the first matching column."""
    if df is None or df.empty:
        return None
    col = _col_ci(df, options)
    if col is None:
        return None
    s = df[col]
    # Try last non-null numeric value
    try:
        s_numeric = s[pd.to_numeric(s, errors="coerce").notna()]
        if not s_numeric.empty:
            return float(s_numeric.iloc[-1])
    except Exception:
        pass
    # Fallback: if already numeric-like
    try:
        return float(s.iloc[-1])
    except Exception:
        return None

# --- Which schemes to consider ---
SCHEMES = ["FiT", "CfD", "Market"]
CATS = SCHEMES + ["lcoe"]  # single LCOE bucket on right axis

# Ensure ext_map exists (build if missing)
if "ext_map" not in globals() or not isinstance(ext_map, dict) or not ext_map:
    ext_map = {}
    for sc in scenarios:
        sid = str(sc.get("scenario_id", ""))
        ov = sc.get("overrides", {})
        ext = ov.get("WindFarm_overrides.external_response_path", None)
        scheme = ov.get("Revenue_overrides.scheme_type", None)
        if not sid or not ext or scheme not in SCHEMES:
            continue
        d = ext_map.setdefault(ext, {})
        d[scheme] = sid

def _get_metrics_for_sid(sid):
    df = RESULTS.get(sid)
    return {
        "NPV":  _extract_metric(df, ("NPV", "net_present_value", "npv")),
        "LCOE": _extract_metric(df, ("LCOE", "lcoe")),
    }

# ------------- Relative figure: ABS deltas -------------
BASELINE_LABEL = "TS_DA_HKN_full_filled_small_gaps_only_TI_boost"

def _find_ext_key_by_label(target_label):
    for ext_path in ext_map.keys():
        label = Path(ext_path).name or str(ext_path)
        if label == target_label:
            return ext_path
    return None

_baseline_key = _find_ext_key_by_label(BASELINE_LABEL)
if _baseline_key is None:
    raise ValueError(f"Baseline '{BASELINE_LABEL}' not found among ext_map keys/labels.")

# Baseline NPV per scheme (left axis)
baseline_npv_by_scheme = {}
for scheme in SCHEMES:
    sid = ext_map[_baseline_key].get(scheme)
    m = _get_metrics_for_sid(sid) if sid else None
    baseline_npv_by_scheme[scheme] = (m or {}).get("NPV", None)

# Helper: pick a case's LCOE (first available across schemes)
def _case_lcoe(schemes_entry):
    for scheme in SCHEMES:
        sid = schemes_entry.get(scheme)
        if not sid:
            continue
        m = _get_metrics_for_sid(sid)
        if m and m.get("LCOE") is not None and not pd.isna(m["LCOE"]):
            return m["LCOE"]
    return None

# Baseline LCOE (right axis)
baseline_lcoe = _case_lcoe(ext_map[_baseline_key])

def _abs_delta(value, base):
    """Absolute delta vs base: value - base."""
    if value is None or base is None:
        return None
    if pd.isna(value) or pd.isna(base):
        return None
    return value - base

def save_fig(fig, filename_base: str, width=1200, height=700, scale=2):
    """
    Save a Plotly figure as PDF and JPEG.
    Requires: pip install -U kaleido
    """

    OUT_DIR = Path(RESULTS_FOLDER)  # e.g. r"M:\Projects\Cost Model\HiperSim\valuewind\results"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for ext in ("pdf", "jpeg"):
        path = OUT_DIR / f"{filename_base}.{ext}"
        try:
            fig.write_image(str(path), format=ext, width=width, height=height, scale=scale)
            print(f"[SAVE] {ext.upper()} → {path}")
        except Exception as e:
            print(f"[WARN] Could not save {ext.upper()} for '{filename_base}': {e}")

from plotly.colors import qualitative


# --- One figure: absolute deltas vs baseline ---
fig_rel = go.Figure()

SCHEMES = ["FiT", "CfD", "Market"]          # assumed defined earlier
CATS = [f"{s} - NPV" for s in SCHEMES] + ["LCoE"]                # x-axis categories (includes single 'lcoe' bucket)

COMMON_MARKER = dict(size=10, symbol="circle")   # same marker for NPV & LCOE
COLORWAY = qualitative.Plotly                   # choose your palette

# OPTIONAL: custom legend texts in the SAME insertion order as ext_map
# If shorter/empty, defaults to the path name label.
LEGEND_TEXTS = [
    "Baseline",
    "Threshold 500 eur",
    "Threshold 800 eur",
    "Threshold 1000 eur",
    "Threshold 1200 eur",
    "Threshold 1400 eur",
    "Threshold 1600 eur",
]

# Preserve insertion order (Python 3.7+ dicts keep it)
items_in_order = list(ext_map.items())

for i, (ext_path, schemes) in enumerate(items_in_order):
    default_label = Path(ext_path).name or str(ext_path)
    legend_name = LEGEND_TEXTS[i] if i < len(LEGEND_TEXTS) and LEGEND_TEXTS[i] else default_label
    group_id = f"resp-{i}"                     # stable legend group id
    color = COLORWAY[i % len(COLORWAY)]        # one color per external response

    # -------- Left axis: ΔNPV (absolute) per scheme vs same-scheme baseline --------
    xs_npv, ys_npv = [], []
    for scheme in SCHEMES:
        sid = schemes.get(scheme)
        if not sid:
            continue
        m = _get_metrics_for_sid(sid)                         # assumes helper exists
        base_npv = baseline_npv_by_scheme.get(scheme)         # assumes dict exists
        delta = _abs_delta(m["NPV"], base_npv)                # assumes helper exists
        if delta is None:
            continue
        #xs_npv.append(str(scheme) + '- NPV')
        xs_npv.append(f"{scheme} - NPV")
        #xs_npv.append(scheme)
        ys_npv.append(delta)

    if xs_npv:
        fig_rel.add_trace(go.Scatter(
            x=xs_npv,
            y=ys_npv,
            mode="markers",
            marker={**COMMON_MARKER, "color": color},          # same marker & color
            name=legend_name,                                  # custom legend text
            legendgroup=group_id,                              # pair with LCOE trace
            showlegend=True,                                   # only this one shows
            legendrank=i,                                      # pin legend position
            hovertemplate="Scheme: %{x}<br>ΔNPV: %{y:,.2f}<extra>" + legend_name + "</extra>",
        ))

    # -------- Right axis: ΔLCOE (absolute) vs baseline case LCOE --------
    lcoe_val = _case_lcoe(schemes)                             # assumes helper exists
    delta_lcoe = _abs_delta(lcoe_val, baseline_lcoe)           # assumes baseline exists
    if delta_lcoe is not None:
        fig_rel.add_trace(go.Scatter(
            x=["LCoE"],                                        # single bucket
            y=[delta_lcoe],
            mode="markers",
            marker={**COMMON_MARKER, "color": color},          # same marker & color as NPV
            name=legend_name,                                  # same name (hidden)
            legendgroup=group_id,
            showlegend=False,                                  # no duplicate legend entry
            yaxis="y2",
            hovertemplate="Metric: LCOE<br>ΔLCOE: %{y:,.4f}<extra>" + legend_name + "</extra>",
        ))

# ---- Layout ----
fig_rel.update_layout(
    title=f"Absolute Δ vs Baseline ({BASELINE_LABEL}) — ΔNPV (left) + ΔLCOE (right)",
    #xaxis_title="Scheme / Metric",
    yaxis_title="ΔNPV to baseline [€]",
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis2=dict(title="ΔLCOE to baseline [€]", overlaying="y", side="right", showgrid=False),
    template="plotly_white",
    width=700,
    height=400,
    #legend_title_text="External response",
    xaxis=dict(categoryorder="array", categoryarray=CATS),
    shapes=[
        dict(type="line", xref="paper", x0=0, x1=1, yref="y",  y0=0, y1=0, line=dict(width=1, dash="dot")),
        dict(type="line", xref="paper", x0=0, x1=1, yref="y2", y0=0, y1=0, line=dict(width=1, dash="dot")),
    ],
    legend=dict(
        traceorder="normal",        # do NOT auto sort legend
        x=1.1, y=1.0,              # move legend to the left
        xanchor="left", yanchor="top",
        groupclick="togglegroup",
    ),
)

fig_rel.show()


save_fig(fig_rel, "delta_vs_baseline")


[WARN] Could not save PDF for 'delta_vs_baseline': 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

[WARN] Could not save JPEG for 'delta_vs_baseline': 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

